1. XGBoost - Optuna

In [ ]:
import optuna
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np

# === 1. Đọc và xử lý dữ liệu ===
df = pd.read_csv('Data/Walmart.csv')
df['Date'] = pd.to_datetime(df['Date'], dayfirst=True)
df['Week'] = df['Date'].dt.isocalendar().week.astype(int)
df['Month'] = df['Date'].dt.month
df['Year'] = df['Date'].dt.year

features = ['Store', 'Holiday_Flag', 'Temperature', 'Fuel_Price', 'CPI',
            'Unemployment', 'Week', 'Month', 'Year']
X = df[features]
y = df['Weekly_Sales']

# === 2. Chia dữ liệu thành train, validation, test ===
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.25, random_state=42)
# → 60% train, 20% val, 20% test

# === 3. Hàm đánh giá ===
def mean_absolute_percentage_error(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    non_zero = y_true != 0
    return np.mean(np.abs((y_true[non_zero] - y_pred[non_zero]) / y_true[non_zero])) * 100

def evaluate_model(name, y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred)
    print(f"\n=== {name} ===")
    print(f"RMSE : {rmse:.2f}")
    print(f"R²   : {r2:.4f}")
    print(f"MAE  : {mae:.2f}")
    print(f"MAPE : {mape:.2f}%")
    return {'Model': name, 'RMSE': rmse, 'R2': r2, 'MAE': mae, 'MAPE': mape}

# === 4. Hàm tối ưu hóa tham số sử dụng tập validation ===
def objective(trial):
    param = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 500),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'gamma': trial.suggest_float('gamma', 0, 0.5),
    }

    model = XGBRegressor(**param, random_state=42)
    model.fit(X_train, y_train)
    y_pred_val = model.predict(X_val)
    rmse = np.sqrt(mean_squared_error(y_val, y_pred_val))
    return rmse

# === 5. Tối ưu hóa bằng Optuna ===
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=50)

print("\n✅ Best hyperparameters:", study.best_params)

# === 6. Huấn luyện mô hình cuối cùng trên toàn bộ tập train + val ===
X_trainval = pd.concat([X_train, X_val])
y_trainval = pd.concat([y_train, y_val])

best_model = XGBRegressor(**study.best_params, random_state=42)
best_model.fit(X_trainval, y_trainval)

# === 7. Đánh giá mô hình trên test set ===
y_pred_test = best_model.predict(X_test)
evaluate_model("XGBoost - Optuna", y_test, y_pred_test)


[I 2025-06-08 12:43:11,643] A new study created in memory with name: no-name-31afa0a5-6012-440e-90d2-25ce8facc841
[I 2025-06-08 12:43:12,296] Trial 0 finished with value: 107888.77490549111 and parameters: {'n_estimators': 328, 'max_depth': 9, 'learning_rate': 0.10576784072180855, 'subsample': 0.69789484443353, 'colsample_bytree': 0.6974812246215075, 'gamma': 0.33405706127989415}. Best is trial 0 with value: 107888.77490549111.
[I 2025-06-08 12:43:12,993] Trial 1 finished with value: 138627.63421667725 and parameters: {'n_estimators': 240, 'max_depth': 11, 'learning_rate': 0.0155175300578858, 'subsample': 0.8525423459448374, 'colsample_bytree': 0.7753627206534222, 'gamma': 0.4803062414518466}. Best is trial 0 with value: 107888.77490549111.
[I 2025-06-08 12:43:13,168] Trial 2 finished with value: 132377.22319582943 and parameters: {'n_estimators': 442, 'max_depth': 4, 'learning_rate': 0.023812207426257215, 'subsample': 0.7705346931500605, 'colsample_bytree': 0.9096157462392784, 'gamma'


✅ Best hyperparameters: {'n_estimators': 464, 'max_depth': 6, 'learning_rate': 0.06518912847270875, 'subsample': 0.6672255041321109, 'colsample_bytree': 0.9682533559841269, 'gamma': 0.46979709189084556}

=== XGBoost - Optuna ===
RMSE : 82321.92
R²   : 0.9790
MAE  : 48956.76
MAPE : 5.17%


{'Model': 'XGBoost - Optuna',
 'RMSE': np.float64(82321.92181514173),
 'R2': 0.9789638280648199,
 'MAE': 48956.75684574106,
 'MAPE': np.float64(5.167171967719239)}

2.Dùng Grid Search tinh chỉnh tham số n_estimators và learning_rate

In [6]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

# Đọc dữ liệu
df = pd.read_csv('Walmart.csv')

# Xử lý ngày tháng
df['Date'] = pd.to_datetime(df['Date'], dayfirst=True)
df['Week'] = df['Date'].dt.isocalendar().week.astype(int)
df['Month'] = df['Date'].dt.month
df['Year'] = df['Date'].dt.year

# 3. Tạo tập đặc trưng và biến mục tiêu
features = ['Store', 'Holiday_Flag', 'Temperature', 'Fuel_Price', 'CPI',
            'Unemployment', 'Week', 'Month', 'Year']
X = df[features]
y = df['Weekly_Sales']

# 4. Tách tập train/test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 5. Hàm đánh giá mô hình
def mean_absolute_percentage_error(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    non_zero = y_true != 0
    return np.mean(np.abs((y_true[non_zero] - y_pred[non_zero]) / y_true[non_zero])) * 100

def evaluate_model(name, y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred)
    print(f"\n=== {name} ===")
    print(f"RMSE : {rmse:.2f}")
    print(f"R²   : {r2:.4f}")
    print(f"MAE  : {mae:.2f}")
    print(f"MAPE : {mape:.2f}%")
    return {'Model': name, 'RMSE': rmse, 'R2': r2, 'MAE': mae, 'MAPE': mape}

# 6. Khởi tạo mô hình XGBoost
xgb = XGBRegressor(random_state=42)

# 7. Tạo không gian tham số cho GridSearchCV
param_grid = {
    'n_estimators': [50, 100, 150],  # Thử 3 giá trị cho n_estimators
    'learning_rate': [0.01, 0.05, 0.1, 0.2]  # Thử 4 giá trị cho learning_rate
}

# 8. Khởi tạo GridSearchCV
grid_search = GridSearchCV(estimator=xgb, param_grid=param_grid, cv=3, scoring='neg_mean_squared_error', n_jobs=-1, verbose=2)

# 9. Chạy GridSearchCV
grid_search.fit(X_train, y_train)

# 10. In ra tham số tốt nhất
print("Best parameters found by GridSearchCV:")
print(grid_search.best_params_)

# 11. Đánh giá mô hình tốt nhất
best_model = grid_search.best_estimator_
y_pred_test = best_model.predict(X_test)

# 12. Đánh giá trên tập test
evaluate_model("XGBoost with GridSearchCV (n_estimators & learning_rate)", y_test, y_pred_test)


Fitting 3 folds for each of 12 candidates, totalling 36 fits
Best parameters found by GridSearchCV:
{'learning_rate': 0.2, 'n_estimators': 150}

=== XGBoost with GridSearchCV (n_estimators & learning_rate) ===
RMSE : 87200.39
R²   : 0.9764
MAE  : 50998.78
MAPE : 5.30%


{'Model': 'XGBoost with GridSearchCV (n_estimators & learning_rate)',
 'RMSE': np.float64(87200.38566375236),
 'R2': 0.9763967110987477,
 'MAE': 50998.7799542298,
 'MAPE': np.float64(5.298327496602268)}

3. Dùng Grid Search để tinh chỉnh 2 tham số tiếp theo là max_deep và min_child_weight

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

# Đọc dữ liệu
df = pd.read_csv('Data/Walmart.csv')

# Xử lý ngày tháng
df['Date'] = pd.to_datetime(df['Date'], dayfirst=True)
df['Week'] = df['Date'].dt.isocalendar().week.astype(int)
df['Month'] = df['Date'].dt.month
df['Year'] = df['Date'].dt.year

# Tạo tập đặc trưng và biến mục tiêu
features = ['Store', 'Holiday_Flag', 'Temperature', 'Fuel_Price', 'CPI',
            'Unemployment', 'Week', 'Month', 'Year']
X = df[features]
y = df['Weekly_Sales']

# Tách tập train/test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Hàm đánh giá mô hình
def mean_absolute_percentage_error(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    non_zero = y_true != 0
    return np.mean(np.abs((y_true[non_zero] - y_pred[non_zero]) / y_true[non_zero])) * 100

def evaluate_model(name, y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred)
    print(f"\n=== {name} ===")
    print(f"RMSE : {rmse:.2f}")
    print(f"R²   : {r2:.4f}")
    print(f"MAE  : {mae:.2f}")
    print(f"MAPE : {mape:.2f}%")
    return {'Model': name, 'RMSE': rmse, 'R2': r2, 'MAE': mae, 'MAPE': mape}


# Định nghĩa GridSearchCV với các tham số muốn tinh chỉnh
param_grid = {
    'max_depth': [3, 5, 7, 9],  # Thử các chiều sâu cây khác nhau
    'min_child_weight': [1, 3, 5, 7]  # Thử các giá trị khác nhau cho min_child_weight
}

# Khởi tạo mô hình XGBoost với các tham số đã xác định trước
xgb = XGBRegressor(learning_rate=0.2, n_estimators=150, random_state=42)

# Cài đặt GridSearchCV
grid_search = GridSearchCV(estimator=xgb, param_grid=param_grid, cv=3, n_jobs=-1, verbose=2)

# Thực hiện Grid Search
grid_search.fit(X_train, y_train)

# In ra tham số tốt nhất
print("Best parameters found by GridSearchCV:")
print(grid_search.best_params_)

# Dự đoán với mô hình tốt nhất
y_pred_test = grid_search.best_estimator_.predict(X_test)

# Đánh giá mô hình trên tập kiểm tra
evaluate_model("XGBoost with GridSearchCV", y_test, y_pred_test)


Fitting 3 folds for each of 16 candidates, totalling 48 fits
Best parameters found by GridSearchCV:
{'max_depth': 7, 'min_child_weight': 5}

=== XGBoost with GridSearchCV ===
RMSE : 86632.70
R²   : 0.9767
MAE  : 50425.40
MAPE : 5.19%


{'Model': 'XGBoost with GridSearchCV',
 'RMSE': np.float64(86632.70460855233),
 'R2': 0.976703029130363,
 'MAE': 50425.39675954255,
 'MAPE': np.float64(5.193440391504935)}

4. Dùng Grid Search để tinh chỉnh 2 tham số tiếp theo là subsample và colorsample_by tree

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

# Đọc dữ liệu
df = pd.read_csv('Data/Walmart.csv')

# Xử lý ngày tháng
df['Date'] = pd.to_datetime(df['Date'], dayfirst=True)
df['Week'] = df['Date'].dt.isocalendar().week.astype(int)
df['Month'] = df['Date'].dt.month
df['Year'] = df['Date'].dt.year

# Tạo tập đặc trưng và biến mục tiêu
features = ['Store', 'Holiday_Flag', 'Temperature', 'Fuel_Price', 'CPI',
            'Unemployment', 'Week', 'Month', 'Year']
X = df[features]
y = df['Weekly_Sales']

# Tách tập train/test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Hàm đánh giá mô hình
def mean_absolute_percentage_error(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    non_zero = y_true != 0
    return np.mean(np.abs((y_true[non_zero] - y_pred[non_zero]) / y_true[non_zero])) * 100

def evaluate_model(name, y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred)
    print(f"\n=== {name} ===")
    print(f"RMSE : {rmse:.2f}")
    print(f"R²   : {r2:.4f}")
    print(f"MAE  : {mae:.2f}")
    print(f"MAPE : {mape:.2f}%")
    return {'Model': name, 'RMSE': rmse, 'R2': r2, 'MAE': mae, 'MAPE': mape}

# Định nghĩa GridSearchCV với các tham số muốn tinh chỉnh
param_grid = {
    'subsample': [0.7, 0.8, 0.9, 1.0],  # Thử các tỷ lệ mẫu khác nhau
    'colsample_bytree': [0.7, 0.8, 0.9, 1.0]  # Thử các tỷ lệ mẫu theo cột khác nhau
}

# Khởi tạo mô hình XGBoost với các tham số đã xác định trước
xgb = XGBRegressor(learning_rate=0.2, n_estimators=150, max_depth=7, min_child_weight=5, random_state=42)

# Cài đặt GridSearchCV
grid_search = GridSearchCV(estimator=xgb, param_grid=param_grid, cv=3, n_jobs=-1, verbose=2)

# Thực hiện Grid Search
grid_search.fit(X_train, y_train)

# In ra tham số tốt nhất
print("Best parameters found by GridSearchCV:")
print(grid_search.best_params_)

# Dự đoán với mô hình tốt nhất
y_pred_test = grid_search.best_estimator_.predict(X_test)

# Đánh giá mô hình trên tập kiểm tra
evaluate_model("XGBoost with GridSearchCV", y_test, y_pred_test)


Fitting 3 folds for each of 16 candidates, totalling 48 fits
Best parameters found by GridSearchCV:
{'colsample_bytree': 1.0, 'subsample': 1.0}

=== XGBoost with GridSearchCV ===
RMSE : 86632.70
R²   : 0.9767
MAE  : 50425.40
MAPE : 5.19%


{'Model': 'XGBoost with GridSearchCV',
 'RMSE': np.float64(86632.70460855233),
 'R2': 0.976703029130363,
 'MAE': 50425.39675954255,
 'MAPE': np.float64(5.193440391504935)}

5. RandomizedSearchCV để tinh chỉnh tham số

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from scipy.stats import uniform

# Đọc dữ liệu
df = pd.read_csv('Data/Walmart.csv')

# Xử lý ngày tháng
df['Date'] = pd.to_datetime(df['Date'], dayfirst=True)
df['Week'] = df['Date'].dt.isocalendar().week.astype(int)
df['Month'] = df['Date'].dt.month
df['Year'] = df['Date'].dt.year

# Tạo tập đặc trưng và biến mục tiêu
features = ['Store', 'Holiday_Flag', 'Temperature', 'Fuel_Price', 'CPI',
            'Unemployment', 'Week', 'Month', 'Year']
X = df[features]
y = df['Weekly_Sales']

# Tách tập train/test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Hàm đánh giá mô hình
def mean_absolute_percentage_error(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    non_zero = y_true != 0
    return np.mean(np.abs((y_true[non_zero] - y_pred[non_zero]) / y_true[non_zero])) * 100

def evaluate_model(name, y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred)
    print(f"\n=== {name} ===")
    print(f"RMSE : {rmse:.2f}")
    print(f"R²   : {r2:.4f}")
    print(f"MAE  : {mae:.2f}")
    print(f"MAPE : {mape:.2f}%")
    return {'Model': name, 'RMSE': rmse, 'R2': r2, 'MAE': mae, 'MAPE': mape}

# Định nghĩa không gian tham số ngẫu nhiên
param_dist = {
    'max_depth': [3, 5, 7, 9, 11],  # Các chiều sâu cây
    'min_child_weight': [1, 3, 5, 7],  # Các giá trị khác nhau cho min_child_weight
    'colsample_bytree': uniform(0.5, 0.5),  # Chọn ngẫu nhiên từ 0.5 đến 1
    'subsample': uniform(0.5, 0.5),  # Chọn ngẫu nhiên từ 0.5 đến 1
    'learning_rate': uniform(0.01, 0.2),  # Chọn ngẫu nhiên từ 0.01 đến 0.2
    'n_estimators': [100, 150, 200, 250]  # Các giá trị cho số cây
}

# Khởi tạo mô hình XGBoost với các tham số đã xác định trước
xgb = XGBRegressor(learning_rate=0.2, n_estimators=150, random_state=42)

# Cài đặt RandomizedSearchCV
random_search = RandomizedSearchCV(
    estimator=xgb, 
    param_distributions=param_dist, 
    n_iter=10,  # Số lượng mẫu tham số ngẫu nhiên
    cv=3,  # Số lần phân chia cross-validation
    verbose=2,  # Cấp độ chi tiết
    n_jobs=-1,  # Sử dụng tất cả các CPU
    random_state=42
)

# Thực hiện tìm kiếm ngẫu nhiên
random_search.fit(X_train, y_train)

# In ra tham số tốt nhất
print("Best parameters found by RandomizedSearchCV:")
print(random_search.best_params_)

# Dự đoán với mô hình tốt nhất
y_pred_test = random_search.best_estimator_.predict(X_test)

# Đánh giá mô hình trên tập kiểm tra
evaluate_model("XGBoost with RandomizedSearchCV", y_test, y_pred_test)


Fitting 3 folds for each of 10 candidates, totalling 30 fits
Best parameters found by RandomizedSearchCV:
{'colsample_bytree': np.float64(0.954660201039391), 'learning_rate': np.float64(0.061755996320003385), 'max_depth': 9, 'min_child_weight': 3, 'n_estimators': 150, 'subsample': np.float64(0.7600340105889054)}

=== XGBoost with RandomizedSearchCV ===
RMSE : 90704.63
R²   : 0.9745
MAE  : 50868.82
MAPE : 4.91%


{'Model': 'XGBoost with RandomizedSearchCV',
 'RMSE': np.float64(90704.63072418443),
 'R2': 0.9744615444323611,
 'MAE': 50868.822949203575,
 'MAPE': np.float64(4.910844967765111)}

6. PSO

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from pyswarm import pso

# === Đọc dữ liệu và xử lý ngày tháng ===
df = pd.read_csv('Data/Walmart.csv')
df['Date'] = pd.to_datetime(df['Date'], dayfirst=True)
df['Week'] = df['Date'].dt.isocalendar().week.astype(int)
df['Month'] = df['Date'].dt.month
df['Year'] = df['Date'].dt.year

# === Lựa chọn đặc trưng ===
features = ['Store', 'Holiday_Flag', 'Temperature', 'Fuel_Price', 'CPI',
            'Unemployment', 'Week', 'Month', 'Year']
X = df[features]
y = df['Weekly_Sales']

# === Tách dữ liệu thành train (64%), val (16%) và test (20%) ===
X_trainval, X_test, y_trainval, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_trainval, y_trainval, test_size=0.2, random_state=42)

# === Hàm MAPE cho đánh giá sau cùng ===
def mean_absolute_percentage_error(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    non_zero = y_true != 0
    return np.mean(np.abs((y_true[non_zero] - y_pred[non_zero]) / y_true[non_zero])) * 100

# === Hàm đánh giá trong quá trình tối ưu hóa ===
def pso_objective(params):
    n_estimators = int(params[0])
    max_depth = int(params[1])
    learning_rate = params[2] / 100  # scale từ [1, 30] → [0.01, 0.30]
    
    model = XGBRegressor(
        n_estimators=n_estimators,
        max_depth=max_depth,
        learning_rate=learning_rate,
        random_state=42,
        verbosity=0
    )
    model.fit(X_train, y_train)
    preds = model.predict(X_val)
    return np.sqrt(mean_squared_error(y_val, preds))  # RMSE

# === Giới hạn cho PSO: [n_estimators, max_depth, learning_rate * 100] ===
lb = [50, 3, 1]     # lower bounds
ub = [300, 15, 30]  # upper bounds

# === Tối ưu tham số với PSO ===
print("Đang tối ưu hóa tham số với PSO...")
best_params, best_rmse = pso(pso_objective, lb, ub, swarmsize=10, maxiter=10)

# === Huấn luyện mô hình cuối cùng với tham số tối ưu ===
n_estimators_opt = int(best_params[0])
max_depth_opt = int(best_params[1])
learning_rate_opt = best_params[2] / 100

final_model = XGBRegressor(
    n_estimators=n_estimators_opt,
    max_depth=max_depth_opt,
    learning_rate=learning_rate_opt,
    random_state=42
)
final_model.fit(X_trainval, y_trainval)  # dùng cả train + val để huấn luyện cuối cùng
final_preds = final_model.predict(X_test)

# === Đánh giá mô hình cuối cùng trên tập test ===
rmse = np.sqrt(mean_squared_error(y_test, final_preds))
mae = mean_absolute_error(y_test, final_preds)
r2 = r2_score(y_test, final_preds)
mape = mean_absolute_percentage_error(y_test, final_preds)

print("\nKẾT QUẢ CUỐI CÙNG TRÊN TEST SET:")
print(f"Tham số tối ưu     : n_estimators={n_estimators_opt}, max_depth={max_depth_opt}, learning_rate={learning_rate_opt:.3f}")
print(f"RMSE               : {rmse:.2f}")
print(f"MAE                : {mae:.2f}")
print(f"R² (R2 Score)      : {r2:.4f}")
print(f"MAPE               : {mape:.2f}%")


Đang tối ưu hóa tham số với PSO...
Stopping search: maximum iterations reached --> 10

KẾT QUẢ CUỐI CÙNG TRÊN TEST SET:
Tham số tối ưu     : n_estimators=250, max_depth=6, learning_rate=0.176
RMSE               : 83177.96
MAE                : 49642.34
R² (R2 Score)      : 0.9785
MAPE               : 5.12%
